# 🎭 GANs Part A: Architecture & Loss Functions

## The GAN Concept

A GAN consists of two networks locked in adversarial training:

- **Generator** (art forger): Takes random noise, produces fake images. Goal: fool the discriminator.
- **Discriminator** (detective): Takes an image (real or fake), outputs a probability of it being real. Goal: catch fakes.

```
  Noise z              Generator               Fake Image
  [100-dim] --------> [G: ConvTranspose] ----> [3x32x32]
                                                    |
  Real Image                                        v
  [3x32x32] -------> [D: Conv] ---------> [P(real)] scalar
                         ^
                         |
                    Adversarial
                      Signal
```

## GenAI Model Family Comparison

| Model | Core Idea | Strengths | Weaknesses |
|---|---|---|---|
| **VAE** | Encode to latent distribution, decode | Smooth latent space, fast sampling | Blurry outputs |
| **GAN** | Adversarial Generator vs Discriminator | Sharp, photorealistic images | Training instability, mode collapse |
| **Diffusion** | Iterative denoising from noise | State-of-the-art quality, diversity | Slow sampling (many steps) |
| **Autoregressive** | Predict next token/pixel sequentially | Excellent for discrete data, text | Slow for images, sequential bottleneck |

In [ ]:
import torch
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super().__init__()
        self.net = nn.Sequential(
            # Block 1: z_dim -> 256 channels, 4x4
            nn.ConvTranspose2d(z_dim, 256, 4, 1, 0), nn.BatchNorm2d(256), nn.ReLU(),
            # Block 2: 256 -> 128 channels, 8x8
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            # Block 3: 128 -> 64 channels, 16x16
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            # Output: 64 -> 3 channels, 32x32
            nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Tanh()
        )
    def forward(self, z):
        return self.net(z)

G = Generator()
noise = torch.randn(1, 100, 1, 1)
fake_img = G(noise)
print(G)
print(f"\nInput noise shape:  {noise.shape}")
print(f"Output image shape: {fake_img.shape}  (batch, channels, H, W)")

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # Block 1: 3 -> 64 channels, 16x16
            nn.Conv2d(3, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            # Block 2: 64 -> 128 channels, 8x8
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            # Block 3: 128 -> 256 channels, 4x4
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2),
            # Output: 256 -> scalar probability
            nn.Conv2d(256, 1, 4, 1, 0), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).view(-1)

D = Discriminator()
img = torch.randn(1, 3, 32, 32)
score = D(img)
print(D)
print(f"\nInput image shape:  {img.shape}")
print(f"Output score shape: {score.shape}  (scalar per image, 0=fake 1=real)")

## Normalization Layers

Normalization stabilizes training by controlling the distribution of activations.

```
Batch (N, C, H, W):
  BatchNorm:    normalize across N (batch dim)  per channel
  LayerNorm:    normalize across C, H, W        per sample
  InstanceNorm: normalize across H, W           per sample per channel
  GroupNorm:    normalize across C/G, H, W      per sample per group
```

| Norm | Normalizes Over | Best For | Notes |
|---|---|---|---|
| **BatchNorm** | Batch + spatial (N, H, W) per channel | GANs (Generator), CNNs | Fails with batch size 1; uses running stats at inference |
| **LayerNorm** | All non-batch dims (C, H, W) per sample | Transformers, NLP | Batch-size independent; stable training |
| **InstanceNorm** | Spatial dims (H, W) per sample per channel | Style transfer, image-to-image | Removes style info, preserves content |
| **GroupNorm** | Groups of channels + spatial per sample | Small batch CNNs, detection | Compromise between BN and LN; batch-size independent |

In [ ]:
import torch
import torch.nn as nn

# Batch: 4 samples, 8 channels, 4x4 spatial
x = torch.randn(4, 8, 4, 4) * 5 + 3  # mean~3, std~5
print(f"Input  — mean: {x.mean():.3f}, std: {x.std():.3f}")

bn = nn.BatchNorm2d(8)
ln = nn.LayerNorm([8, 4, 4])

x_bn = bn(x)
x_ln = ln(x)

# BatchNorm: per-channel stats
print(f"\nBatchNorm  — per-channel mean: {x_bn.mean(dim=[0,2,3]).detach().numpy().round(3)}")
print(f"BatchNorm  — per-channel std:  {x_bn.std(dim=[0,2,3]).detach().numpy().round(3)}")

# LayerNorm: per-sample stats
print(f"\nLayerNorm  — per-sample mean: {x_ln.mean(dim=[1,2,3]).detach().numpy().round(3)}")
print(f"LayerNorm  — per-sample std:  {x_ln.std(dim=[1,2,3]).detach().numpy().round(3)}")

In [ ]:
import torch
import torch.nn as nn

G = Generator()  # from cell 3
D = Discriminator()  # from cell 4
bce = nn.BCELoss()
batch_size = 8

# --- Fake batch from Generator ---
z = torch.randn(batch_size, 100, 1, 1)
fake_imgs = G(z).detach()          # detach: don't backprop through G yet
real_imgs = torch.randn(batch_size, 3, 32, 32)  # stand-in for real data

# --- Discriminator Loss ---
d_real = D(real_imgs)              # D's scores on real images
d_fake = D(fake_imgs)              # D's scores on fake images
loss_d_real = bce(d_real, torch.ones(batch_size))   # real -> label 1
loss_d_fake = bce(d_fake, torch.zeros(batch_size))  # fake -> label 0
loss_D = (loss_d_real + loss_d_fake) / 2

# --- Generator Loss (modified minimax: maximize log D(G(z))) ---
fake_imgs_g = G(z)                 # fresh forward pass for G
d_fake_g = D(fake_imgs_g)
loss_G = bce(d_fake_g, torch.ones(batch_size))  # G wants D to output 1 on fakes

print(f"D(real) mean: {d_real.mean():.4f}  |  D(fake) mean: {d_fake.mean():.4f}")
print(f"Loss D (real): {loss_d_real:.4f}  |  Loss D (fake): {loss_d_fake:.4f}")
print(f"Loss D total:  {loss_D:.4f}")
print(f"Loss G:        {loss_G:.4f}")

## Quiz

**Q1: Why do GANs use LeakyReLU in the Discriminator but ReLU in the Generator?**

<details>
<summary>Show Answer</summary>

LeakyReLU allows small negative gradients to flow through the Discriminator, preventing dead neurons when inputs are negative. This is critical because the Discriminator sees many real and fake inputs with varied activations. The Generator uses ReLU (or Tanh at the end) since it is building up features from scratch with BatchNorm stabilizing the distributions, and dead neurons are less of a problem there.

</details>

---

**Q2: What is mode collapse in GANs, and why does it happen?**

<details>
<summary>Show Answer</summary>

Mode collapse is when the Generator learns to produce only a small subset of possible outputs (e.g., always the same face) because it found a single mode that consistently fools the Discriminator. It happens because the Generator's objective only cares about fooling D — not about diversity. The Discriminator eventually catches up and rejects that mode, causing the Generator to jump to another single mode, cycling rather than covering the full data distribution.

</details>

---

**Q3: Why does the modified minimax loss (maximize log D(G(z))) work better than the original minimax (minimize log(1 - D(G(z)))) for training the Generator?**

<details>
<summary>Show Answer</summary>

Early in training, when the Generator is weak, D(G(z)) is close to 0. The gradient of `log(1 - D(G(z)))` is very small near 0, causing vanishing gradients and slow Generator learning. Maximizing `log D(G(z))` instead produces large gradients when D(G(z)) is near 0 (because log approaches -inf), giving the Generator a strong training signal precisely when it needs it most.

</details>